In [ ]:
# Python 3.10에서 진행합니다. (터미널 열기 귀찮아서...)
import time
import undetected_chromedriver as uc
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.keys import Keys
import re

import pandas as pd

In [ ]:
import platform
print(platform.machine())

In [ ]:
# 웹드라이버 실행
options = uc.ChromeOptions()
options.add_argument(f'--user-data-dir=/Users/koreanraichu/Library/Application Support/Google/Chrome/Default')
# 본인의 크롬 설치 경로 확인 (보통 아래와 같습니다)
driver = uc.Chrome(options=options, browser_executable_path='/Applications/Google Chrome.app/Contents/MacOS/Google Chrome')

In [ ]:
# 사이트 오우픈
driver.get('https://www.the-b.co/reviews?brand=rextreme')

# 로그인은 수동으로 합니다..

In [ ]:
df = {
    '감정':[],
    '아이디':[],
    '날짜':[],
    '본문':[],
    '해시태그':[],
    '좋아요':[],
    '댓글':[]
}

In [ ]:
# 시작 인덱스 설정
current_idx = 1
consecutive_failures = 0 # 연속 실패 횟수 (종료 조건 체크용)

print("🚀 날짜 범위 수집을 시작합니다. (필터링된 카드를 순차적으로 긁습니다)")

while True:
    try:
        # 1. 현재 순서의 카드 찾기 (5열 격자 순차 접근)
        card_xpath = f'/html/body/div[contains(@class, "min-h-screen")]/main/div/div[contains(@class, "grid")]/div[{current_idx}]/div'
        card = driver.find_element(By.XPATH, value=card_xpath)

        # 2. 화면 중앙으로 스크롤 (데이터 로딩 유도)
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", card)
        time.sleep(0.3)

        # --- 데이터 추출 로직 ---
        title = card.find_element(By.XPATH, './div[1]')
        content = card.find_element(By.XPATH, './div[2]')

        emo_text = title.find_element(By.XPATH, './div[2]/span[2]').text
        user_id = content.find_element(By.XPATH, './div[1]/div/span').text
        raw_date = content.find_element(By.XPATH, './div[1]/span').text

        # 상세 내용 확인을 위한 버튼 클릭
        right_button = content.find_element(By.XPATH, './div[2]/button[2]')
        right_button.click()
        time.sleep(0.2)

        body_text = content.find_element(By.XPATH, './p').text
        spans = content.find_elements(By.XPATH, './div[3]//span')
        tags_list = [s.text for s in spans if s.text.strip()]

        # 좋아요/댓글 추출 (정규식 re 사용)
        stats = content.find_elements(By.XPATH, ".//span[contains(@class, 'items-center')]")
        def parse_stat(el):
            val = re.sub(r'[^0-9]', '', el.text)
            return int(val) if val else 0

        likes = parse_stat(stats[0]) if len(stats) > 0 else 0
        comments = parse_stat(stats[1]) if len(stats) > 1 else 0

        # --- df에 추가 ---
        df['감정'].append(emo_text)
        df['아이디'].append(user_id)
        df['날짜'].append(raw_date)
        df['본문'].append(body_text)
        df['해시태그'].append(tags_list)
        df['좋아요'].append(likes)
        df['댓글'].append(comments)

        print(f"✅ [{current_idx}] {user_id} ({raw_date}) 완료")

        # 성공 시 인덱스 증가 및 실패 카운트 초기화
        current_idx += 1
        consecutive_failures = 0

        # 3. 15개(3줄)마다 한 번씩 바닥으로 스크롤해서 새 데이터 로딩 유도
        if current_idx % 15 == 0:
            print("⏳ 15개 단위 스크롤 및 추가 로딩 대기...")
            driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.END)
            time.sleep(2)

    except Exception as e:
        # 요소를 못 찾으면(스크롤 끝 도달) 잠시 기다렸다가 재시도
        consecutive_failures += 1
        print(f"⚠️ {current_idx}번 확인 중... (재시도 {consecutive_failures}/3)")

        # 화면을 조금 더 내려보고 다시 시도
        driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.PAGE_DOWN)
        time.sleep(2)

        # 3번 연속 실패하면 정말 끝인 것으로 간주하고 루프 탈출
        if consecutive_failures >= 3:
            print("🏁 더 이상 표시되는 카물이 없습니다. 수집을 종료합니다.")
            break

# 최종 결과 확인
import pandas as pd
result_df = pd.DataFrame(df)
print(f"총 {len(result_df)}개의 데이터를 수집했습니다.")

In [ ]:
df = pd.DataFrame(df)
df.to_csv('Site fork_3.csv', index=False, encoding='utf-8-sig')
driver.quit()